# Seeing PCA — Projection is Just Casting a Shadow

> *"It's not 'can you read the music', it's 'can you hear it'."*

You already know the mechanics: standardize → covariance → eigenvectors → multiply. This notebook is about **hearing the music** — actually *seeing* what "projecting onto a principal component" does to your data.

**The one idea that unlocks everything:**

> Projection = shining a light on your data cloud and looking at the **shadow** it casts on a line (or a plane).

PCA's whole job is to find the angle to shine the light so the shadow keeps the most detail.

We'll build this in 2D first (where you can literally see every step), then 3D (rotatable), then connect it back to the 4D Iris data where you *can't* draw it but now you'll know what's happening.

---

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(42)

# PART 1 — 2D: Where You Can See Everything

Let's make up the simplest possible example: **study hours vs exam score** for 120 students. These two are correlated — students who study more tend to score higher. So the data cloud will be *stretched along a diagonal*.

In [15]:
# Make correlated 2D data: study hours vs exam score
n = 120
hidden = np.random.uniform(-3, 3, n)   # a hidden 'overall ability' factor

study_hours = hidden * 1.0 + np.random.normal(0, 0.4, n)
exam_score  = hidden * 0.5 + np.random.normal(0, 0.4, n)

X = np.column_stack([study_hours, exam_score])
print("First 5 samples (before centering):")
print(X[:5].round(2))
X = X - X.mean(axis=0)   # center it (PCA always works from the origin)

print(f"Data shape: {X.shape}  (120 students, 2 features)")
print(f"First 5 points:\n{X[:5].round(2)}")

First 5 samples (before centering):
[[ 0.18 -0.09]
 [ 1.79  1.33]
 [ 1.07  0.67]
 [ 3.54  2.51]
 [-2.17 -0.89]]
Data shape: (120, 2)  (120 students, 2 features)
First 5 points:
[[ 0.11 -0.14]
 [ 1.73  1.28]
 [ 1.    0.62]
 [ 3.48  2.46]
 [-2.24 -0.94]]


In [3]:
# Just LOOK at the cloud first
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=X[:, 0], y=X[:, 1], mode='markers',
    marker=dict(size=8, color='#534AB7', opacity=0.6),
    name='students'
))
fig.update_layout(
    title='Our data cloud — notice it is stretched along a diagonal',
    xaxis_title='study hours (centered)',
    yaxis_title='exam score (centered)',
    width=600, height=600
)
# equal aspect ratio so directions are not distorted
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.show()

**See the diagonal stretch?** The cloud isn't a round blob — it's an ellipse tilted up-and-to-the-right. That tilt is the *real* pattern: there's basically one underlying thing (call it "overall ability") driving both features.

PCA is going to *discover* that diagonal direction on its own.

## Step 1: Find the principal components (the natural axes of the cloud)

In [4]:
# Covariance + eigendecomposition (you know this part)
cov = np.cov(X, rowvar=False)
eigvals, eigvecs = np.linalg.eigh(cov)

# sort descending
order = np.argsort(eigvals)[::-1]
eigvals = eigvals[order]
eigvecs = eigvecs[:, order]

pc1 = eigvecs[:, 0]   # direction of MOST spread
pc2 = eigvecs[:, 1]   # direction of LEAST spread (perpendicular to pc1)

print(f"PC1 direction: {pc1.round(3)}   (eigenvalue {eigvals[0]:.2f} — big spread)")
print(f"PC2 direction: {pc2.round(3)}   (eigenvalue {eigvals[1]:.2f} — tiny spread)")
print(f"\nPC1 carries {100*eigvals[0]/eigvals.sum():.1f}% of the variance")

PC1 direction: [-0.908 -0.418]   (eigenvalue 3.91 — big spread)
PC2 direction: [ 0.418 -0.908]   (eigenvalue 0.14 — tiny spread)

PC1 carries 96.6% of the variance


In [5]:
# Draw the PC arrows ON the cloud — these are the 'natural axes' PCA found
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=X[:, 0], y=X[:, 1], mode='markers',
    marker=dict(size=7, color='#534AB7', opacity=0.4), name='students'
))

# scale arrows by sqrt(eigenvalue) so arrow length = how much spread in that direction
s1 = 2.5 * np.sqrt(eigvals[0])
s2 = 2.5 * np.sqrt(eigvals[1])

fig.add_trace(go.Scatter(
    x=[0, pc1[0]*s1], y=[0, pc1[1]*s1], mode='lines+markers',
    line=dict(color='#D85A30', width=5), name='PC1 (most spread)'
))
fig.add_trace(go.Scatter(
    x=[0, pc2[0]*s2], y=[0, pc2[1]*s2], mode='lines+markers',
    line=dict(color='#1D9E75', width=5), name='PC2 (least spread)'
))

fig.update_layout(
    title='PCA found the cloud\'s natural axes. PC1 points along the stretch.',
    xaxis_title='study hours', yaxis_title='exam score',
    width=650, height=600
)
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.show()

**This is the key realization.** PCA didn't pick the x-axis or the y-axis. It picked the *orange* direction — the one that runs straight down the length of the cloud. That's PC1. PC2 is perpendicular to it (the green one), pointing across the thin part of the cloud.

The arrow *lengths* show the eigenvalues: PC1 is long (lots of spread), PC2 is stubby (barely any spread).

## Step 2: NOW the magic — projection is casting a shadow

"Projecting onto PC1" means: imagine PC1 is a wall, and a light shines perpendicular onto it. Each data point casts a shadow onto that wall. **Where the shadow lands is the point's new 1D coordinate.**

Let's draw the shadow lines explicitly.

In [6]:
# Project every point onto the PC1 line
# scalar coordinate along PC1 = dot product of point with pc1 unit vector
coords_pc1 = X @ pc1               # shape (120,) — the 1D shadow positions

# where each shadow lands back in 2D space (the 'foot' of the perpendicular)
feet = np.outer(coords_pc1, pc1)   # shape (120, 2)

fig = go.Figure()

# the PC1 line (extended)
line_ext = 3.2 * np.sqrt(eigvals[0])
fig.add_trace(go.Scatter(
    x=[-pc1[0]*line_ext, pc1[0]*line_ext],
    y=[-pc1[1]*line_ext, pc1[1]*line_ext],
    mode='lines', line=dict(color='#D85A30', width=3), name='PC1 line (the wall)'
))

# thin perpendicular 'light ray' lines from each point to its shadow
for i in range(n):
    fig.add_trace(go.Scatter(
        x=[X[i, 0], feet[i, 0]], y=[X[i, 1], feet[i, 1]],
        mode='lines', line=dict(color='gray', width=0.5),
        opacity=0.4, showlegend=False, hoverinfo='skip'
    ))

# original points
fig.add_trace(go.Scatter(
    x=X[:, 0], y=X[:, 1], mode='markers',
    marker=dict(size=7, color='#534AB7', opacity=0.55), name='original point'
))
# the shadows (feet) on the line
fig.add_trace(go.Scatter(
    x=feet[:, 0], y=feet[:, 1], mode='markers',
    marker=dict(size=6, color='#D85A30'), name='shadow on PC1'
))

fig.update_layout(
    title='Each gray ray drops a point onto PC1. The orange dots are the shadows.',
    xaxis_title='study hours', yaxis_title='exam score',
    width=700, height=650
)
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.show()

**THIS is projection.** Every purple point sends a perpendicular ray down to the orange line. The orange dot where it lands is its shadow. We just turned a 2D point into a single number (how far along the orange line the shadow sits).

Notice the shadows are nicely spread out along the line — they keep most of the separation the original points had. That's *not an accident*. PCA chose this line precisely because it maximizes how spread out the shadows are.

## Step 3: Why is PC1 the *best* line? Compare a bad choice.

Let's project onto a bad direction (say, straight down onto the x-axis) and watch the shadows bunch up and lose information.

In [7]:
bad_dir = np.array([1.0, 0.0])    # project onto x-axis (a bad choice)
coords_bad = X @ bad_dir
feet_bad = np.outer(coords_bad, bad_dir)

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    f'GOOD: project onto PC1<br>shadow spread (variance) = {coords_pc1.var():.2f}',
    f'BAD: project onto x-axis<br>shadow spread (variance) = {coords_bad.var():.2f}'
))

# --- left: good ---
fig.add_trace(go.Scatter(x=[-pc1[0]*line_ext, pc1[0]*line_ext],
    y=[-pc1[1]*line_ext, pc1[1]*line_ext], mode='lines',
    line=dict(color='#D85A30', width=3), showlegend=False), row=1, col=1)
for i in range(0, n, 2):
    fig.add_trace(go.Scatter(x=[X[i,0], feet[i,0]], y=[X[i,1], feet[i,1]],
        mode='lines', line=dict(color='gray', width=0.5), opacity=0.3,
        showlegend=False, hoverinfo='skip'), row=1, col=1)
fig.add_trace(go.Scatter(x=X[:,0], y=X[:,1], mode='markers',
    marker=dict(size=5, color='#534AB7', opacity=0.5), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=feet[:,0], y=feet[:,1], mode='markers',
    marker=dict(size=5, color='#D85A30'), showlegend=False), row=1, col=1)

# --- right: bad ---
fig.add_trace(go.Scatter(x=[-line_ext, line_ext], y=[0, 0], mode='lines',
    line=dict(color='#888', width=3), showlegend=False), row=1, col=2)
for i in range(0, n, 2):
    fig.add_trace(go.Scatter(x=[X[i,0], feet_bad[i,0]], y=[X[i,1], feet_bad[i,1]],
        mode='lines', line=dict(color='gray', width=0.5), opacity=0.3,
        showlegend=False, hoverinfo='skip'), row=1, col=2)
fig.add_trace(go.Scatter(x=X[:,0], y=X[:,1], mode='markers',
    marker=dict(size=5, color='#534AB7', opacity=0.5), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=feet_bad[:,0], y=feet_bad[:,1], mode='markers',
    marker=dict(size=5, color='#888'), showlegend=False), row=1, col=2)

fig.update_layout(title='PC1 keeps the shadows spread out. The bad line squishes them together.',
    width=950, height=500)
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.show()

**The whole point of PCA in one picture.** Both sides flatten 2D down to 1D. But:
- Left (PC1): the shadows stay spread out → points that were different *stay* different → we kept the information.
- Right (x-axis): the shadows bunch together → points that were different now overlap → we lost information.

"Maximize variance" literally means "keep the shadows as spread out as possible." That number under each title — the variance of the shadows — is exactly the eigenvalue. PC1 wins because its shadow variance is the highest possible.

## Step 4: The 1D result — what we actually hand to the next algorithm

In [8]:
# The projected data is just those shadow positions — now 1D numbers
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=coords_pc1, y=np.zeros(n), mode='markers',
    marker=dict(size=8, color=coords_pc1, colorscale='Viridis',
                showscale=True, colorbar=dict(title='PC1 value')),
    name='students on the PC1 number line'
))
fig.update_layout(
    title='Our 120 students, now described by ONE number each (position on PC1)',
    xaxis_title='PC1 coordinate (≈ "overall ability")',
    yaxis=dict(showticklabels=False, range=[-1, 1]),
    width=800, height=300
)
fig.show()

print("We went from 2 numbers per student to 1 number, keeping",
      f"{100*eigvals[0]/eigvals.sum():.1f}% of the information.")
print("\nFirst 5 students' single PC1 coordinate:", coords_pc1[:5].round(2))

We went from 2 numbers per student to 1 number, keeping 96.6% of the information.

First 5 students' single PC1 coordinate: [ 0.81 -3.13 -1.48 -0.38  2.21]


Each student is now ONE number. That number is essentially their "overall ability" — PCA distilled two correlated measurements into the single underlying thing driving them.

## Step 5: Reconstruction — putting the shadow back, and what we lost

In [9]:
# Reconstruct: take the 1D shadow and place it back in 2D space
# (this is just 'feet' — the shadow positions ARE the reconstruction)
X_reconstructed = np.outer(coords_pc1, pc1)

fig = go.Figure()
# error lines (what we threw away = distance from point to its shadow)
for i in range(n):
    fig.add_trace(go.Scatter(
        x=[X[i,0], X_reconstructed[i,0]], y=[X[i,1], X_reconstructed[i,1]],
        mode='lines', line=dict(color='#E24B4A', width=0.8),
        opacity=0.5, showlegend=False, hoverinfo='skip'))
fig.add_trace(go.Scatter(x=X[:,0], y=X[:,1], mode='markers',
    marker=dict(size=7, color='#534AB7', opacity=0.5), name='original (2D)'))
fig.add_trace(go.Scatter(x=X_reconstructed[:,0], y=X_reconstructed[:,1], mode='markers',
    marker=dict(size=6, color='#D85A30'), name='reconstructed (from 1D)'))

fig.update_layout(
    title='Red lines = the information we lost (the PC2 part we dropped)',
    xaxis_title='study hours', yaxis_title='exam score',
    width=700, height=650)
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.show()

mse = np.mean(np.sum((X - X_reconstructed)**2, axis=1))
print(f"Reconstruction error (avg squared red-line length): {mse:.3f}")
print(f"This equals the dropped eigenvalue: {eigvals[1]:.3f}")

Reconstruction error (avg squared red-line length): 0.138
This equals the dropped eigenvalue: 0.139


Those little red lines ARE the lost information — the spread in the PC2 direction we decided to ignore. They're short, because the cloud was thin in that direction. **The reconstruction error equals exactly the eigenvalue we dropped.** That's not a coincidence; it's the definition.

---

# PART 2 — 3D: Projecting Onto a Plane

In 3D, PC1 + PC2 define a *plane*. Projecting means dropping every 3D point onto that flat plane — like pressing a 3D object flat. Drag the plot to rotate and see the plane sitting inside the cloud.

In [10]:
# Make 3D data that mostly lives on a tilted plane (3rd direction is thin)
n3 = 200
a = np.random.uniform(-3, 3, n3)
b = np.random.uniform(-2, 2, n3)
X3 = np.column_stack([
    a + 0.3*b + np.random.normal(0, 0.25, n3),
    0.5*a - b + np.random.normal(0, 0.25, n3),
    0.4*a + 0.6*b + np.random.normal(0, 0.25, n3)
])
X3 = X3 - X3.mean(axis=0)

cov3 = np.cov(X3, rowvar=False)
vals3, vecs3 = np.linalg.eigh(cov3)
o = np.argsort(vals3)[::-1]; vals3 = vals3[o]; vecs3 = vecs3[:, o]

print("3D eigenvalues:", vals3.round(2))
print("Variance kept by top 2 PCs:",
      f"{100*vals3[:2].sum()/vals3.sum():.1f}%")

3D eigenvalues: [4.87 1.84 0.06]
Variance kept by top 2 PCs: 99.1%


In [11]:
# Build the PC1-PC2 plane as a mesh
u1, u2 = vecs3[:, 0], vecs3[:, 1]
gr = np.linspace(-4, 4, 10)
ss, tt = np.meshgrid(gr, gr)
plane = (ss[..., None]*u1 + tt[..., None]*u2)

# project points onto the plane
coords_2d = X3 @ np.column_stack([u1, u2])   # (n,2) coords in plane
feet3 = coords_2d @ np.column_stack([u1, u2]).T  # back in 3D

fig = go.Figure()
fig.add_trace(go.Surface(
    x=plane[:,:,0], y=plane[:,:,1], z=plane[:,:,2],
    opacity=0.25, colorscale=[[0,'#D85A30'],[1,'#D85A30']],
    showscale=False, name='PC1-PC2 plane'))
fig.add_trace(go.Scatter3d(
    x=X3[:,0], y=X3[:,1], z=X3[:,2], mode='markers',
    marker=dict(size=3, color='#534AB7', opacity=0.7), name='original 3D'))
fig.add_trace(go.Scatter3d(
    x=feet3[:,0], y=feet3[:,1], z=feet3[:,2], mode='markers',
    marker=dict(size=2.5, color='#D85A30'), name='shadow on plane'))

fig.update_layout(
    title='3D cloud + the plane PCA chose. Drag to rotate!',
    width=750, height=650, scene=dict(aspectmode='data'))
fig.show()

**Rotate it.** You'll see the orange plane is angled to slice straight through the fattest part of the cloud — just like the PC1 line did in 2D, but now it's a 2D plane in 3D space. The orange dots are the shadows pressed onto that plane.

Now here's the payoff: those shadows live on a flat plane, so we can describe each one with just 2 numbers (position on the plane). Let's flatten it and look.

In [12]:
# The 2D view = looking straight at the plane head-on
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=coords_2d[:, 0], y=coords_2d[:, 1], mode='markers',
    marker=dict(size=6, color=coords_2d[:, 0], colorscale='Viridis', opacity=0.7)))
fig.update_layout(
    title='Same data, now flattened to 2D (looking at the plane head-on)',
    xaxis_title='PC1', yaxis_title='PC2', width=650, height=550)
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.show()

print("We squashed 3D → 2D, keeping",
      f"{100*vals3[:2].sum()/vals3.sum():.1f}% of the information.")

We squashed 3D → 2D, keeping 99.1% of the information.


That flattening — taking the shadow on the plane and looking at it head-on — is *exactly* what `X_pca = X_std @ W` does. The matrix multiply IS the act of pressing the cloud onto the plane and reading off the 2 in-plane coordinates.

---

# PART 3 — Back to Iris (4D): Same Music, More Instruments

In 4D you can't draw the arrows or the plane — but nothing conceptually changes:
- The cloud lives in 4D.
- PCA finds the 4 perpendicular natural axes (eigenvectors).
- We keep the top 2 (the plane where the shadow is most spread out).
- `X @ W` casts the 4D shadow onto that 2D plane.

The only difference from Part 2 is that you can't *see* the original cloud — but you trust it works because you just watched it work in 2D and 3D.

In [13]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

iris = load_iris()
Xi = StandardScaler().fit_transform(iris.data)
yi = iris.target
names = iris.target_names

covi = np.cov(Xi, rowvar=False)
vi, Vi = np.linalg.eigh(covi)
oi = np.argsort(vi)[::-1]; vi = vi[oi]; Vi = Vi[:, oi]

W = Vi[:, :2]                # the 'plane' we project onto
Xi_pca = Xi @ W             # cast the 4D shadow onto the 2D plane

fig = go.Figure()
colors = ['#534AB7', '#D85A30', '#1D9E75']
for k in range(3):
    m = yi == k
    fig.add_trace(go.Scatter(
        x=Xi_pca[m, 0], y=Xi_pca[m, 1], mode='markers',
        marker=dict(size=8, color=colors[k], opacity=0.7), name=names[k]))
fig.update_layout(
    title=f'Iris 4D shadow cast onto 2D ({100*vi[:2].sum()/vi.sum():.1f}% variance kept)',
    xaxis_title='PC1', yaxis_title='PC2', width=700, height=550)
fig.show()

## Finally: that reconstruction scatter plot you asked about

The plot below (the one that confused you) checks reconstruction quality, one feature at a time. Here's how to *read* it:

- **X-axis** = a feature's true standardized value.
- **Y-axis** = what we get back after squashing to 2D and un-squashing (reconstruction).
- **Red dashed line** = the "perfect" line where reconstructed exactly equals original.
- **A dot ON the line** = that feature was perfectly preserved. A dot far off the line = that feature lost info.

It's just plotting *truth vs recovered* for each feature. The tighter the dots hug the red line, the less PCA distorted that feature.

In [14]:
# Reconstruct Iris from 2 PCs, then compare per feature
Xi_recon = Xi_pca @ W.T
fnames = iris.feature_names

fig = make_subplots(rows=2, cols=2, subplot_titles=fnames)
for i, fname in enumerate(fnames):
    r, c = (i // 2) + 1, (i % 2) + 1
    fig.add_trace(go.Scatter(
        x=Xi[:, i], y=Xi_recon[:, i], mode='markers',
        marker=dict(size=5, opacity=0.5,
                    color=[colors[t] for t in yi]),
        showlegend=False), row=r, col=c)
    lo = min(Xi[:, i].min(), Xi_recon[:, i].min())
    hi = max(Xi[:, i].max(), Xi_recon[:, i].max())
    fig.add_trace(go.Scatter(
        x=[lo, hi], y=[lo, hi], mode='lines',
        line=dict(color='#E24B4A', dash='dash', width=1.5),
        showlegend=False), row=r, col=c)

fig.update_layout(
    title='Truth (x) vs Reconstructed (y). On the red line = perfectly kept.',
    height=650, width=750)
fig.show()

print("Sepal width dots scatter most off the line —")
print("that feature lived partly in PC3/PC4, which we dropped.")

Sepal width dots scatter most off the line —
that feature lived partly in PC3/PC4, which we dropped.


---

# The Music, In Words

Now you can *hear* it:

1. **Your data is a cloud** with a shape — stretched more in some directions than others.
2. **Eigenvectors are the cloud's natural axes** — the directions it's stretched along.
3. **Projection = casting a shadow** onto those axes (a line in 2D, a plane in 3D, a hyperplane in 4D+).
4. **`X @ W` is literally the shadow-casting operation** — that matrix multiply presses the cloud onto the chosen axes and reads the new coordinates.
5. **PCA picks the axes where the shadow stays most spread out** (max variance = max eigenvalue), so flattening loses the least.
6. **The directions we drop** had thin shadows anyway — that's the small reconstruction error.

When you write `PCA(n_components=2).fit_transform(X)`, you're saying: *"shine a light on my data and hand me the shadow on the 2-axis plane that keeps the most detail."*

That's the whole song. Next stop whenever you're ready: **LDA** — same shadow-casting idea, but it picks the angle that best *separates the labeled classes* instead of just maximizing spread.